# LeetCode 907 — Sum of Subarray Minimums bằng Python

Link đề: https://leetcode.com/problems/sum-of-subarray-minimums/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Stack, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def sumSubarrayMins(self, arr: List[int]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên, lấy **modulo `10^9 + 7`**). Điểm cốt lõi: đổi cách đếm — thay vì duyệt mọi đoạn con, tính **mỗi phần tử là giá trị nhỏ nhất của bao nhiêu đoạn con** (kỹ thuật **đóng góp**) bằng **ngăn xếp đơn điệu**; xử lý các phần tử **bằng nhau** bằng quy ước "bên trái `<`, bên phải `≤`" để mỗi đoạn con chỉ tính một lần. Oracle: **vét cạn mọi đoạn con** với `min(arr[i:j+1])`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tổng `min(b)` trên **mọi đoạn con liên tiếp** `b` của `arr`, lấy modulo `10^9 + 7`.
2. Nắm công cụ Python: ngăn xếp `list` các chỉ số, `min` trên lát cắt, số nguyên lớn không tràn (nhưng vẫn phải `% MOD` theo đề), mảng `left`/`right` biên.
3. Đi từ **vét cạn `O(n²)` với min chạy** → **hai lượt ngăn xếp tính biên trái/phải** → **một lượt ngăn xếp, tính đóng góp khi pop** → **quy hoạch động theo điểm cuối**.
4. Hiểu mấu chốt: với mỗi `arr[i]`, số đoạn con nhận nó làm giá trị nhỏ nhất là `(i − L[i]) · (R[i] − i)`, trong đó `L[i]` là chỉ số gần nhất bên trái có giá trị **nhỏ hơn hẳn** và `R[i]` là chỉ số gần nhất bên phải có giá trị **nhỏ hơn hoặc bằng** (quy ước bất đối xứng để không đếm trùng khi có phần tử bằng nhau).
5. Tránh lỗi kinh điển: dùng cả hai bên `<` (đếm trùng) hoặc cả hai bên `≤` (đếm thiếu), biên mặc định sai (`0` thay vì `-1`/`n`), quên `% MOD`, và tràn số nguyên 32-bit ở ngôn ngữ khác.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`. Tìm tổng của `min(b)` với `b` chạy trên **mọi đoạn con liên tiếp** của `arr`. Vì đáp án có thể rất lớn, trả về nó **modulo `10^9 + 7`**.

**Ví dụ 1:** `arr = [3,1,2,4]` → `17`. Các đoạn con: `[3],[1],[2],[4],[3,1],[1,2],[2,4],[3,1,2],[1,2,4],[3,1,2,4]` có min `3,1,2,4,1,1,2,1,1,1`, tổng `17`.

**Ví dụ 2:** `arr = [11,81,94,43,3]` → `444`.

**Ràng buộc:** `1 <= arr.length <= 3·10^4`; `1 <= arr[i] <= 3·10^4`.

In [1]:
import itertools
import random
import time
from typing import List

MOD = 10 ** 9 + 7
examples = [([3, 1, 2, 4], 17), ([11, 81, 94, 43, 3], 444)]
for a, want in examples:
    print(f"arr = {a} -> {want}")

arr = [3, 1, 2, 4] -> 17
arr = [11, 81, 94, 43, 3] -> 444


## 3. Kiến thức nền

### 3.1 Kỹ thuật đóng góp (contribution technique)

Thay vì cộng theo từng đoạn con, cộng theo từng **phần tử**: tổng `= Σ_i arr[i] · cnt[i]` với `cnt[i]` là số đoạn con mà `arr[i]` là phần tử nhỏ nhất. Đây là hoán đổi thứ tự lấy tổng, giảm từ `O(n²)` đoạn xuống `n` phần tử.

### 3.2 Biên và số đoạn con

Cho `L` = chỉ số gần nhất bên trái của `i` mà `arr[L] < arr[i]` (hoặc `−1`), `R` = chỉ số gần nhất bên phải mà `arr[R] ≤ arr[i]` (hoặc `n`). Một đoạn con `[l, r]` có `arr[i]` là min khi `L < l ≤ i ≤ r < R`: `(i − L)` cách chọn `l`, `(R − i)` cách chọn `r`. Vậy `cnt[i] = (i − L)(R − i)`.

### 3.3 Vì sao `<` một bên và `≤` bên kia

Nếu một đoạn có **nhiều phần tử bằng nhau cùng là min**, phải chọn đúng **một** làm "chủ sở hữu" để không đếm trùng. Quy ước `L: <` và `R: ≤`: phần tử `i` có thể mở rộng sang **trái** qua các phần tử bằng nó (vì `L` chỉ dừng ở giá trị nhỏ hơn hẳn) nhưng **không** mở rộng sang phải qua phần tử bằng nó (vì `R` dừng ở giá trị `≤`). Vậy `i` sở hữu đoạn khi nó là phần tử min **bên phải nhất**. Mỗi đoạn có đúng một min bên phải nhất ⇒ được đếm đúng một lần.

### 3.4 Ngăn xếp đơn điệu tìm biên

Ngăn xếp chỉ số có giá trị tăng chặt. Duyệt trái sang phải: pop khi `arr[top] ≥ arr[i]`; sau pop, đỉnh còn lại (nếu có) là `L` của `i` (phần tử gần nhất `< arr[i]`). Với mỗi phần tử bị pop `j` bởi `i`: `R[j] = i` (phần tử đầu tiên bên phải có `≤ arr[j]`).

In [2]:
def boundaries(arr):
    n = len(arr)
    L, R = [-1] * n, [n] * n
    st = []
    for i, x in enumerate(arr):
        while st and arr[st[-1]] >= x:
            R[st.pop()] = i                              # i là phần tử đầu tiên bên phải có giá trị <= arr[j]
        L[i] = st[-1] if st else -1                      # đỉnh còn lại là phần tử gần nhất bên trái có giá trị < x
        st.append(i)
    return L, R


arr = [3, 1, 2, 4]
L, R = boundaries(arr)
print("L =", L, "| R =", R, "| cnt =", [(i - L[i]) * (R[i] - i) for i in range(4)])

L = [-1, -1, 1, 2] | R = [1, 4, 4, 4] | cnt = [1, 6, 2, 1]


## 4. Ý tưởng nền tảng / trực giác

`arr = [3, 1, 2, 4]`. Với mỗi phần tử, đếm số đoạn con nhận nó làm min:

| `i` | `arr[i]` | `L` (trái, `<`) | `R` (phải, `≤`) | `(i − L)(R − i)` | đóng góp |
|---|---|---|---|---|---|
| 0 | 3 | −1 | 1 (vì `1 ≤ 3`) | `1 · 1 = 1` | 3 |
| 1 | 1 | −1 | 4 (không có) | `2 · 3 = 6` | 6 |
| 2 | 2 | 1 (`1 < 2`) | 4 | `1 · 2 = 2` | 4 |
| 3 | 4 | 2 (`2 < 4`) | 4 | `1 · 1 = 1` | 4 |

Tổng `3 + 6 + 4 + 4 = 17` ✓.

Phần tử `1` là min của `6` đoạn: `[1], [3,1], [1,2], [3,1,2], [1,2,4], [3,1,2,4]`.

In [3]:
arr = [3, 1, 2, 4]
n = len(arr)
total = 0
for i in range(n):
    owns = [(l, r) for l in range(n) for r in range(l, n) if l <= i <= r and min(arr[l:r + 1]) == arr[i] and arr.index(min(arr[l:r + 1])) == i]
    total += arr[i] * len(owns)
    print(f"arr[{i}]={arr[i]}: min của {len(owns)} đoạn {[arr[l:r + 1] for l, r in owns]}")
print("tổng =", total)

arr[0]=3: min của 1 đoạn [[3]]
arr[1]=1: min của 6 đoạn [[3, 1], [3, 1, 2], [3, 1, 2, 4], [1], [1, 2], [1, 2, 4]]
arr[2]=2: min của 2 đoạn [[2], [2, 4]]
arr[3]=4: min của 1 đoạn [[4]]
tổng = 17


## 5. Cách 1 — Brute Force: mọi đoạn con, min chạy

**Ý tưởng:** hai vòng lặp: điểm đầu `i`, điểm cuối `j ≥ i`, duy trì `cur_min = min(cur_min, arr[j])`, cộng vào tổng.

In [4]:
def sum_mins_brute(arr):
    total = 0
    for i in range(len(arr)):
        cur = arr[i]
        for j in range(i, len(arr)):
            cur = min(cur, arr[j])                       # min của arr[i..j]
            total += cur
    return total % MOD


for a, want in examples:
    assert sum_mins_brute(a) == want
print([sum_mins_brute(a) for a, _ in examples])

[17, 444]


In [5]:
# Code nộp LeetCode cho Cách 1 (O(n^2): 4.5·10^8 phép với n = 3·10^4, quá chậm trong Python)
class SolutionBrute:
    def sumSubarrayMins(self, arr: List[int]) -> int:
        total = 0
        for i in range(len(arr)):
            cur = arr[i]
            for j in range(i, len(arr)):
                cur = min(cur, arr[j])
                total += cur
        return total % (10 ** 9 + 7)

**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ.

**Nút thắt:** ta cộng từng đoạn riêng lẻ; nhiều đoạn có cùng min và có thể đếm theo nhóm.

## 6. Ý tưởng thuật toán / bước đột phá

Đảo thứ tự lấy tổng: `Σ_đoạn min = Σ_i arr[i] · cnt[i]` với `cnt[i] = (i − L[i]) · (R[i] − i)`. Tìm `L`, `R` bằng ngăn xếp đơn điệu trong `O(n)`.

Cách tiếp cận khác, theo điểm cuối: `dp[i]` = tổng min của các đoạn **kết thúc tại `i`**; `dp[i] = dp[L[i]] + arr[i] · (i − L[i])` (với `dp[−1] = 0`). Tổng đáp án `= Σ dp[i]`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** hai lượt ngăn xếp tính `L`, `R` rồi đếm, **Cách 3** một lượt ngăn xếp (tính khi pop), **Cách 4** quy hoạch động theo điểm cuối.

### Cách 2 — Hai ngăn xếp: biên trái `<`, biên phải `≤`

**Giải thích từng đoạn:**

1. Lượt trái→phải: ngăn xếp chỉ số, pop khi `arr[top] ≥ arr[i]`; `left[i] = top` (hoặc `−1`).
2. Lượt phải→trái: pop khi `arr[top] > arr[i]`; `right[i] = top` (hoặc `n`) — là phần tử đầu tiên bên phải có giá trị **`≤`**... chú ý: pop khi `>` để dừng ở phần tử `≤`.
3. `ans = Σ arr[i] · (i − left[i]) · (right[i] − i)`, lấy `mod`.

In [6]:
def sum_mins_two_stacks(arr):
    n = len(arr)
    left, right = [-1] * n, [n] * n
    st = []
    for i in range(n):                                   # biên trái: gần nhất có giá trị < arr[i]
        while st and arr[st[-1]] >= arr[i]:
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    st = []
    for i in range(n - 1, -1, -1):                       # biên phải: gần nhất có giá trị <= arr[i]
        while st and arr[st[-1]] > arr[i]:
            st.pop()
        right[i] = st[-1] if st else n
        st.append(i)
    return sum(arr[i] * (i - left[i]) * (right[i] - i) for i in range(n)) % MOD


for a, want in examples:
    assert sum_mins_two_stacks(a) == want
print([sum_mins_two_stacks(a) for a, _ in examples])

[17, 444]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionTwoStacks:
    def sumSubarrayMins(self, arr: List[int]) -> int:
        n = len(arr)
        left, right = [-1] * n, [n] * n
        st = []
        for i in range(n):
            while st and arr[st[-1]] >= arr[i]:
                st.pop()
            left[i] = st[-1] if st else -1
            st.append(i)
        st = []
        for i in range(n - 1, -1, -1):
            while st and arr[st[-1]] > arr[i]:
                st.pop()
            right[i] = st[-1] if st else n
            st.append(i)
        return sum(arr[i] * (i - left[i]) * (right[i] - i) for i in range(n)) % (10 ** 9 + 7)

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 3 — Một lượt ngăn xếp: tính khi pop

**Ý tưởng:** duyệt `i = 0..n` với một phần tử "gác" `arr[n] = 0` (nhỏ hơn mọi giá trị) để pop hết. Khi `i` pop phần tử `j` (do `arr[j] ≥ arr[i]`): `R[j] = i` và `L[j]` là đỉnh mới sau pop (hoặc `−1`) — tính ngay `arr[j] · (j − L) · (i − j)` và cộng.

In [8]:
def sum_mins_one_pass(arr):
    total = 0
    st = []                                              # chỉ số, giá trị tăng chặt
    for i in range(len(arr) + 1):
        cur = arr[i] if i < len(arr) else 0              # phần tử gác 0 ép pop hết
        while st and arr[st[-1]] >= cur:
            j = st.pop()
            left = st[-1] if st else -1                  # gần nhất bên trái có giá trị < arr[j]
            total += arr[j] * (j - left) * (i - j)       # i là gần nhất bên phải có giá trị <= arr[j]
        st.append(i)
    return total % MOD


for a, want in examples:
    assert sum_mins_one_pass(a) == want
print([sum_mins_one_pass(a) for a, _ in examples])

[17, 444]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionOnePass:
    def sumSubarrayMins(self, arr: List[int]) -> int:
        total, st = 0, []
        for i in range(len(arr) + 1):
            cur = arr[i] if i < len(arr) else 0
            while st and arr[st[-1]] >= cur:
                j = st.pop()
                left = st[-1] if st else -1
                total += arr[j] * (j - left) * (i - j)
            st.append(i)
        return total % (10 ** 9 + 7)

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ, một lượt. (Chỉ số `n` được đẩy vào cuối nhưng không dùng; phần tử gác `0` nhỏ hơn mọi `arr[j] ≥ 1`.)

### Cách 4 — Quy hoạch động theo điểm cuối

**Ý tưởng:** `dp[i]` = tổng min của các đoạn con **kết thúc tại `i`**. Gọi `p` = chỉ số gần nhất bên trái có `arr[p] < arr[i]` (hoặc `−1`):

- các đoạn bắt đầu ở `p + 1 … i` có min `= arr[i]` (mọi phần tử giữa `≥ arr[i]`): `(i − p)` đoạn;
- các đoạn bắt đầu ở `≤ p` có min `= min của đoạn đó cắt tại p` (vì `arr[p] < arr[i] ≤` các phần tử giữa), nên tổng bằng `dp[p]`.

`dp[i] = dp[p] + arr[i] · (i − p)`; đáp án `Σ dp[i]`.

In [10]:
def sum_mins_dp(arr):
    n = len(arr)
    dp = [0] * n
    st = []
    total = 0
    for i in range(n):
        while st and arr[st[-1]] >= arr[i]:
            st.pop()
        p = st[-1] if st else -1                         # gần nhất bên trái có giá trị < arr[i]
        dp[i] = (dp[p] if p >= 0 else 0) + arr[i] * (i - p)
        total += dp[i]
        st.append(i)
    return total % MOD


for a, want in examples:
    assert sum_mins_dp(a) == want
print([sum_mins_dp(a) for a, _ in examples])

[17, 444]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionDP:
    def sumSubarrayMins(self, arr: List[int]) -> int:
        n = len(arr)
        dp, st, total = [0] * n, [], 0
        for i in range(n):
            while st and arr[st[-1]] >= arr[i]:
                st.pop()
            p = st[-1] if st else -1
            dp[i] = (dp[p] if p >= 0 else 0) + arr[i] * (i - p)
            total += dp[i]
            st.append(i)
        return total % (10 ** 9 + 7)

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ; không cần biên phải.

## 8. TRACE từng bước

**Trace Cách 3** với `arr = [3, 1, 2, 4]` (gác `0` ở cuối):

| `i` | `cur` | pop `j` | `left` | `(j − left)·(i − j)` | cộng `arr[j]·…` | ngăn xếp sau |
|---|---|---|---|---|---|---|
| 0 | 3 | — | — | — | — | `[0]` |
| 1 | 1 | `0` (`3 ≥ 1`) | −1 | `1·1 = 1` | `3` | `[1]` |
| 2 | 2 | — | — | — | — | `[1, 2]` |
| 3 | 4 | — | — | — | — | `[1, 2, 3]` |
| 4 (gác) | 0 | `3` (4≥0) | 2 | `1·1 = 1` | `4` | |
| | | `2` (2≥0) | 1 | `1·2 = 2` | `4` | |
| | | `1` (1≥0) | −1 | `2·3 = 6` | `6` | `[4]` |

Tổng `3 + 4 + 4 + 6 = 17` ✓. Hàm trace:

In [12]:
def trace_one_pass(arr):
    total, st = 0, []
    print(f"arr = {arr}")
    print(f"{'i':>2} {'cur':>4}  pop j  left  (j-left)*(i-j)  cộng   ngăn xếp sau")
    for i in range(len(arr) + 1):
        cur = arr[i] if i < len(arr) else 0
        lines = []
        while st and arr[st[-1]] >= cur:
            j = st.pop()
            left = st[-1] if st else -1
            add = arr[j] * (j - left) * (i - j)
            total += add
            lines.append((j, left, (j - left) * (i - j), add))
        st.append(i)
        if not lines:
            print(f"{i:>2} {cur:>4}  {'—':>5}  {'—':>4}  {'—':>14}  {'—':>4}   {st}")
        for k, (j, left, cnt, add) in enumerate(lines):
            tail = f"   {st}" if k == len(lines) - 1 else ""
            print(f"{i:>2} {cur:>4}  {j:>5}  {left:>4}  {cnt:>14}  {add:>4}{tail}")
    print("=> tổng =", total)
    return total


assert trace_one_pass([3, 1, 2, 4]) == 17

arr = [3, 1, 2, 4]
 i  cur  pop j  left  (j-left)*(i-j)  cộng   ngăn xếp sau
 0    3      —     —               —     —   [0]
 1    1      0    -1               1     3   [1]
 2    2      —     —               —     —   [1, 2]
 3    4      —     —               —     —   [1, 2, 3]
 4    0      3     2               1     4
 4    0      2     1               2     4
 4    0      1    -1               6     6   [4]
=> tổng = 17


**Trace Cách 4:** `[3,1,2,4]`: `dp[0] = 3`; `p` của `1` là `−1` ⇒ `dp[1] = 0 + 1·2 = 2`; `p` của `2` là `1` ⇒ `dp[2] = dp[1] + 2·1 = 4`; `p` của `4` là `2` ⇒ `dp[3] = dp[2] + 4·1 = 8`. Tổng `3 + 2 + 4 + 8 = 17` ✓.

## 9. Code nộp LeetCode

Chọn **Cách 3** (một lượt ngăn xếp). Giữ nguyên `class Solution`, `sumSubarrayMins(self, arr)`:

In [13]:
class Solution:
    def sumSubarrayMins(self, arr: List[int]) -> int:
        MOD = 10 ** 9 + 7
        total, st = 0, []                                 # st: chỉ số có giá trị tăng chặt
        for i in range(len(arr) + 1):
            cur = arr[i] if i < len(arr) else 0           # phần tử gác 0 để pop hết ngăn xếp
            while st and arr[st[-1]] >= cur:
                j = st.pop()                              # arr[j] là min của các đoạn trong (left, i), chứa j
                left = st[-1] if st else -1               # gần nhất bên trái có giá trị < arr[j]
                total += arr[j] * (j - left) * (i - j)    # (j - left) cách chọn đầu, (i - j) cách chọn cuối
            st.append(i)
        return total % MOD


print([Solution().sumSubarrayMins(a) for a, _ in examples])        # [17, 444]
assert all(Solution().sumSubarrayMins(a) == want for a, want in examples)

[17, 444]


## 10. Vì sao đúng

**(a) Đóng góp.** `Σ_{đoạn} min(đoạn) = Σ_i arr[i] · #{đoạn mà i là "chủ sở hữu" của min}`, trong đó mỗi đoạn có đúng **một** chủ sở hữu.

**(b) Quy ước chủ sở hữu duy nhất.** Với một đoạn `[l, r]`, gọi `m` là giá trị nhỏ nhất và `i*` là chỉ số **bên phải nhất** trong đoạn có giá trị `m`. Đặt `L(i) =` gần nhất bên trái có giá trị `< arr[i]`, `R(i) =` gần nhất bên phải có giá trị `≤ arr[i]`. Khi đó `i` là chủ sở hữu của `[l, r]` ⇔ `L(i) < l ≤ i ≤ r < R(i)`:
- (⇒) `i = i*`: trái `i` trong đoạn mọi phần tử `≥ m` nên `L(i) < l`; phải `i` trong đoạn mọi phần tử `> m` (vì `i*` là bên phải nhất) nên `R(i) > r`.
- (⇐) nếu `L(i) < l` và `r < R(i)` thì trong đoạn mọi phần tử bên trái `i` là `≥ arr[i]` và mọi phần tử bên phải là `> arr[i]`, nên min là `arr[i]` và `i` là bên phải nhất ⇒ `i = i*`.
Số đoạn của `i` là `(i − L(i))(R(i) − i)`.

**(c) Ngăn xếp tìm `L`, `R`.** Bất biến: trước khi xử lý `i`, ngăn xếp chứa các chỉ số `j < i` mà mọi phần tử giữa `j` và `i` (không kể hai đầu) đều `> arr[j]`, giá trị tăng chặt từ đáy lên đỉnh. Khi `i` đến, pop mọi `j` có `arr[j] ≥ arr[i]`. Với `j` bị pop bởi `i`: các phần tử giữa `j` và `i` đều `> arr[j]` (theo bất biến) và `arr[i] ≤ arr[j]`, nên `i` chính là phần tử đầu tiên bên phải `j` có giá trị `≤ arr[j]`, tức `R[j] = i`. Sau khi pop xong, đỉnh còn lại `t` thoả `arr[t] < arr[i]` và mọi phần tử giữa `t` và `i` đã bị pop (đều `≥ arr[i]`), nên `t = L[i]`.

**(d) Cách 3** là cách 2 với `R[j] = i` ngay khi `j` bị pop, và phần tử gác `0` bảo đảm pop hết còn lại với `R = n`.

**(e) Cách 4.** Đã chứng minh ở mục 7: các đoạn kết thúc tại `i` bắt đầu ở `(p, i]` có min `arr[i]`; các đoạn bắt đầu ở `≤ p` có cùng min với phần tương ứng kết thúc tại `p`, do `arr[p] < arr[i] ≤` mọi phần tử trong `(p, i]`.

**(f) Modulo.** Đáp án là tổng số nguyên, lấy `mod` một lần ở cuối (Python không tràn). Ở ngôn ngữ 32-bit cần `long` và `% MOD` sau mỗi phép nhân/cộng.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **mọi đoạn con** trên mọi mảng nhỏ kể cả có phần tử bằng nhau (mục 12D), và khớp nhau trên `n = 3·10^4` với kiểm tra công thức đóng (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi đoạn con | `O(n²)` | `O(1)` | oracle, `n` nhỏ |
| 2. Hai ngăn xếp | `O(n)` | `O(n)` | rõ ràng |
| 3. Một lượt ngăn xếp | `O(n)` | `O(n)` | **đáp án chính** |
| 4. DP theo điểm cuối | `O(n)` | `O(n)` | một biên |

Đo thực tế (`n = 3·10^4`; cách 1 chỉ `n = 2000`):

In [14]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(907)
n = 30000
datasets = {
    "ngẫu nhiên": [rng.randint(1, 30000) for _ in range(n)],
    "tăng dần": list(range(1, n + 1)),
    "giảm dần": list(range(n, 0, -1)),
    "toàn bằng nhau": [30000] * n,
}
for label, a in datasets.items():
    row = [f"{name} {bench(fn, a) * 1e3:7.2f} ms" for name, fn in
           [("2stacks", sum_mins_two_stacks), ("onepass", sum_mins_one_pass), ("dp", sum_mins_dp)]]
    print(f"{label:16s}: " + " | ".join(row))
print(f"cách 1 với n = 2000: {bench(sum_mins_brute, datasets['ngẫu nhiên'][:2000], 1) * 1e3:7.1f} ms (tăng theo n²)")

ngẫu nhiên      : 2stacks   17.13 ms | onepass   11.58 ms | dp   11.57 ms


tăng dần        : 2stacks   13.73 ms | onepass   10.01 ms | dp    7.89 ms
giảm dần        : 2stacks   10.47 ms | onepass    8.55 ms | dp    8.25 ms


toàn bằng nhau  : 2stacks   13.39 ms | onepass    8.26 ms | dp    8.91 ms


cách 1 với n = 2000:   152.3 ms (tăng theo n²)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ (kể cả bằng nhau và hoán vị) với oracle `min(arr[i:j+1])`, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn (`n = 3·10^4`, công thức đóng, kiểm tra `mod`).

### A. Ví dụ của đề

In [15]:
def oracle(arr):
    """Đúng định nghĩa: cộng min(arr[i..j]) cho mọi đoạn con, rồi lấy modulo."""
    n = len(arr)
    return sum(min(arr[i:j + 1]) for i in range(n) for j in range(i, n)) % MOD


IMPLS = {
    "brute": sum_mins_brute,
    "two_stacks": sum_mins_two_stacks,
    "one_pass": sum_mins_one_pass,
    "dp": sum_mins_dp,
    "Solution": lambda a: Solution().sumSubarrayMins(list(a)),
    "SolutionBrute": lambda a: SolutionBrute().sumSubarrayMins(list(a)),
    "SolutionTwoStacks": lambda a: SolutionTwoStacks().sumSubarrayMins(list(a)),
    "SolutionOnePass": lambda a: SolutionOnePass().sumSubarrayMins(list(a)),
    "SolutionDP": lambda a: SolutionDP().sumSubarrayMins(list(a)),
}


def check(arr, want=None, names=None):
    if want is None:
        want = oracle(arr)
    for name in (names or IMPLS):
        got = IMPLS[name](list(arr))
        assert got == want, (name, arr, got, want)


for a, want in examples:
    assert oracle(a) == want
    check(a, want)
print("ví dụ đề: oracle mọi-đoạn-con và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle mọi-đoạn-con và 9 cài đặt đúng


### B. Edge case

- Một phần tử: tổng `= arr[0]`.
- Hai phần tử bằng nhau: `[2,2]` → `6` (đoạn `[2],[2],[2,2]`).
- Mọi phần tử bằng nhau: `x·n(n+1)/2`.
- Tăng dần / giảm dần.
- Giá trị nhỏ nhất ở đầu / giữa / cuối.
- Nhiều giá trị nhỏ nhất bằng nhau.
- Giá trị lớn nhất `3·10^4` (để kiểm tra modulo sau).

In [16]:
cases = [
    ([5], 5), ([2, 2], 6), ([1, 1, 1], 6), ([3, 3, 3, 3], 30), ([1, 2, 3], 10), ([3, 2, 1], 10),
    ([1, 5, 5, 1], 1 + 5 + 5 + 1 + 1 + 5 + 1 + 1 + 1 + 1), ([2, 1, 2, 1], None), ([4, 1, 4], None), ([30000] * 3, 30000 * 6),
]
for a, want in cases:
    check(a, want)
assert Solution().sumSubarrayMins([2, 2]) == 6
assert Solution().sumSubarrayMins([30000] * 30000) == (30000 * (30000 * 30001 // 2)) % MOD
print(len(cases), "edge case +2: đúng")

10 edge case +2: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [17]:
def bug_both_strict(arr):                              # biên trái < và biên phải < : đếm trùng khi có phần tử bằng nhau
    n = len(arr)
    left, right = [-1] * n, [n] * n
    st = []
    for i in range(n):
        while st and arr[st[-1]] >= arr[i]:
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    st = []
    for i in range(n - 1, -1, -1):
        while st and arr[st[-1]] >= arr[i]:            # SAI: pop khi >=, nên biên phải là gần nhất có giá trị < (nghiêm ngặt)
            st.pop()
        right[i] = st[-1] if st else n
        st.append(i)
    return sum(arr[i] * (i - left[i]) * (right[i] - i) for i in range(n)) % MOD


def bug_both_nonstrict(arr):                           # cả hai biên <= : đếm thiếu khi có phần tử bằng nhau
    n = len(arr)
    left, right = [-1] * n, [n] * n
    st = []
    for i in range(n):
        while st and arr[st[-1]] > arr[i]:             # SAI: biên trái dừng ở giá trị <= (không nghiêm ngặt)
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    st = []
    for i in range(n - 1, -1, -1):
        while st and arr[st[-1]] > arr[i]:
            st.pop()
        right[i] = st[-1] if st else n
        st.append(i)
    return sum(arr[i] * (i - left[i]) * (right[i] - i) for i in range(n)) % MOD


def bug_default_zero(arr):                             # biên mặc định 0 thay vì -1 (chỉ số trước đoạn đầu)
    total, st = 0, []
    for i in range(len(arr) + 1):
        cur = arr[i] if i < len(arr) else 0
        while st and arr[st[-1]] >= cur:
            j = st.pop()
            left = st[-1] if st else 0                 # SAI: phải là -1
            total += arr[j] * (j - left) * (i - j)
        st.append(i)
    return total % MOD


def bug_no_mod(arr):                                   # quên modulo
    return Solution_nomod(arr)


def Solution_nomod(arr):
    total, st = 0, []
    for i in range(len(arr) + 1):
        cur = arr[i] if i < len(arr) else 0
        while st and arr[st[-1]] >= cur:
            j = st.pop()
            left = st[-1] if st else -1
            total += arr[j] * (j - left) * (i - j)
        st.append(i)
    return total                                       # SAI: thiếu % MOD


def bug_int32(arr):                                    # mô phỏng int 32-bit tràn khi nhân ở ngôn ngữ khác
    def wrap(v):
        return ((v + 2 ** 31) % 2 ** 32) - 2 ** 31

    total, st = 0, []
    for i in range(len(arr) + 1):
        cur = arr[i] if i < len(arr) else 0
        while st and arr[st[-1]] >= cur:
            j = st.pop()
            left = st[-1] if st else -1
            total = wrap(total + wrap(arr[j] * (j - left) * (i - j)))
        st.append(i)
    return total % MOD


def bug_dp_no_carry(arr):                              # DP quên cộng dp[p]: chỉ tính các đoạn bắt đầu trong (p, i]
    n = len(arr)
    st, total = [], 0
    for i in range(n):
        while st and arr[st[-1]] >= arr[i]:
            st.pop()
        p = st[-1] if st else -1
        total += arr[i] * (i - p)                      # SAI: thiếu dp[p]
        st.append(i)
    return total % MOD


BUGS = [bug_both_strict, bug_both_nonstrict, bug_default_zero, bug_dp_no_carry]
rng = random.Random(3)
tests = [[rng.randint(1, 4) for _ in range(rng.randint(1, 7))] for _ in range(600)]
for bug in BUGS:
    bad = next((a for a in tests if bug(a) != oracle(a)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:18s} sai trên {bad}: ra {bug(bad)}, đúng {oracle(bad)}")
big = [30000] * 30000
print("bug_no_mod (n = 3·10^4, toàn 30000):", bug_no_mod(big), "| đúng:", Solution().sumSubarrayMins(big))
print("bug_int32 :", bug_int32(big), "| đúng:", Solution().sumSubarrayMins(big))
assert bug_no_mod(big) != Solution().sumSubarrayMins(big) and bug_int32(big) != Solution().sumSubarrayMins(big)

bug_both_strict    sai trên [4, 1, 1, 4, 3]: ra 31, đúng 25
bug_both_nonstrict sai trên [4, 1, 1, 4, 3]: ra 19, đúng 25
bug_default_zero   sai trên [2, 3]: ra 3, đúng 7
bug_dp_no_carry    sai trên [2, 3]: ra 5, đúng 7


bug_no_mod (n = 3·10^4, toàn 30000): 13500450000000 | đúng: 449905500
bug_int32 : 367788665 | đúng: 449905500


### D. Vét cạn mọi mảng nhỏ

(1) Mọi mảng độ dài `1..6` với giá trị `1..4` (`4 + 16 + 64 + 256 + 1 024 + 4 096 = 5 460`) — có nhiều phần tử bằng nhau; (2) mọi **hoán vị** của `1..7` (`5 040`) — mọi giá trị khác nhau. Oracle: `min(arr[i:j+1])` cho mọi đoạn con.

In [18]:
t0 = time.perf_counter()
cnt = 0
fast = ("brute", "two_stacks", "one_pass", "dp", "Solution")
for n in range(1, 7):
    for tup in itertools.product(range(1, 5), repeat=n):
        a = list(tup)
        want = oracle(a)
        for name in fast:
            assert IMPLS[name](a) == want, (name, a)
        cnt += 1
for perm in itertools.permutations(range(1, 8)):
    a = list(perm)
    want = oracle(a)
    for name in fast:
        assert IMPLS[name](a) == want, (name, a)
    cnt += 1
print(cnt, "mảng nhỏ: 5 cài đặt khớp oracle mọi-đoạn-con;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 5460 + 5040

10500 mảng nhỏ: 5 cài đặt khớp oracle mọi-đoạn-con; 0.3 s


### E. Ngẫu nhiên có seed và tính chất

Mảng ngẫu nhiên tới `300` phần tử so với Cách 1 (và oracle `O(n³)` tới `25`). Tính chất: (1) đảo mảng **không đổi** đáp án; (2) nhân mọi phần tử với `c` nhân đáp án với `c` (trước modulo); (3) cộng `c` vào mọi phần tử tăng tổng đúng `c · n(n+1)/2`; (4) tổng `Σ cnt[i] = n(n+1)/2` (mỗi đoạn một chủ sở hữu).

In [19]:
def exact_sum(a):
    """Tổng chính xác (không modulo) bằng quy hoạch động O(n) trên số nguyên lớn của Python."""
    dp_prev, st, total = None, [], 0
    dp = [0] * len(a)
    for i, x in enumerate(a):
        while st and a[st[-1]] >= x:
            st.pop()
        p = st[-1] if st else -1
        dp[i] = (dp[p] if p >= 0 else 0) + x * (i - p)
        total += dp[i]
        st.append(i)
    return total


rng = random.Random(20240907)
for _ in range(1500):
    n = rng.randint(1, 300)
    hi = rng.choice([3, 20, 30000])
    a = [rng.randint(1, hi) for _ in range(n)]
    want = oracle(a) if n <= 25 else sum_mins_brute(a)
    for name in ("two_stacks", "one_pass", "dp", "Solution", "SolutionTwoStacks", "SolutionOnePass", "SolutionDP"):
        assert IMPLS[name](a) == want, (name, a)
    assert Solution().sumSubarrayMins(a[::-1]) == want                                             # (1) đảo mảng
    c = rng.randint(2, 5)
    assert exact_sum([c * x for x in a]) == c * exact_sum(a)                                       # (2) nhân với c
    assert exact_sum([x + c for x in a]) == exact_sum(a) + c * n * (n + 1) // 2                    # (3) cộng c
    L, R = boundaries(a)
    assert sum((i - L[i]) * (R[i] - i) for i in range(n)) == n * (n + 1) // 2                      # (4) mỗi đoạn một chủ sở hữu
print("1500 mảng ngẫu nhiên: 7 cài đặt khớp; 4 tính chất (đảo, nhân c, cộng c, Σ cnt = n(n+1)/2) đúng")

1500 mảng ngẫu nhiên: 7 cài đặt khớp; 4 tính chất (đảo, nhân c, cộng c, Σ cnt = n(n+1)/2) đúng


### F. Giới hạn: `n = 3·10^4`

Các cách `O(n)` khớp nhau; công thức đóng cho các dạng đặc biệt (so sánh với số nguyên chính xác không modulo rồi mới lấy `mod`), để chắc chắn phép modulo đúng.

In [20]:
rng = random.Random(500)
n = 30000
fast = ["two_stacks", "one_pass", "dp", "Solution", "SolutionTwoStacks", "SolutionOnePass", "SolutionDP"]
datasets = {
    "ngẫu nhiên": [rng.randint(1, 30000) for _ in range(n)],
    "tăng dần": list(range(1, n + 1)),
    "giảm dần": list(range(n, 0, -1)),
    "toàn bằng nhau": [30000] * n,
    "răng cưa": [1 if i % 2 else 30000 for i in range(n)],
    "chỉ 2 giá trị": [rng.choice([1, 2]) for _ in range(n)],
}
for name, a in datasets.items():
    res = {k: IMPLS[k](a) for k in fast}
    assert len(set(res.values())) == 1, (name, res)
    assert res["one_pass"] == exact_sum(a) % MOD
    print(f"{name:16s}: {res['one_pass']:>10}  (tổng chính xác lớn hơn MOD: {exact_sum(a) >= MOD})")
assert Solution().sumSubarrayMins(datasets["toàn bằng nhau"]) == (30000 * n * (n + 1) // 2) % MOD
assert Solution().sumSubarrayMins(datasets["tăng dần"]) == sum(i * (n - i + 1) for i in range(1, n + 1)) % MOD
assert Solution().sumSubarrayMins(datasets["giảm dần"]) == sum(x * (idx + 1) for idx, x in enumerate(range(n, 0, -1))) % MOD
print("công thức đóng khớp (toàn bằng nhau, tăng dần, giảm dần)")

ngẫu nhiên      :  976280302  (tổng chính xác lớn hơn MOD: True)


tăng dần        :  449978500  (tổng chính xác lớn hơn MOD: True)
giảm dần        :  449978500  (tổng chính xác lớn hơn MOD: True)


toàn bằng nhau  :  449905500  (tổng chính xác lớn hơn MOD: True)


răng cưa        :  900000000  (tổng chính xác lớn hơn MOD: False)


chỉ 2 giá trị   :  450045546  (tổng chính xác lớn hơn MOD: False)
công thức đóng khớp (toàn bằng nhau, tăng dần, giảm dần)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: hai biên cùng `<` — đếm trùng khi có giá trị bằng nhau

In [21]:
print("[2,2]: hai biên <:", bug_both_strict([2, 2]), "| đúng:", Solution().sumSubarrayMins([2, 2]))
assert bug_both_strict([2, 2]) == 8 and Solution().sumSubarrayMins([2, 2]) == 6

[2,2]: hai biên <: 8 | đúng: 6


**Vì sao sai:** đoạn `[2,2]` có hai phần tử min bằng nhau; nếu cả hai cùng "nhận" đoạn đó thì bị cộng hai lần. **Đúng:** một bên `<`, một bên `≤`.

### Lỗi 2: hai biên cùng `≤` — đếm thiếu

In [22]:
print("[2,2]: hai biên <=:", bug_both_nonstrict([2, 2]), "| đúng:", Solution().sumSubarrayMins([2, 2]))
assert bug_both_nonstrict([2, 2]) == 4

[2,2]: hai biên <=: 4 | đúng: 6


**Vì sao sai:** đoạn `[2,2]` không có chủ sở hữu nào (mỗi phần tử bị chặn bởi phần tử bằng nó ở cả hai phía). **Đúng:** bất đối xứng.

### Lỗi 3: biên mặc định `0` thay vì `-1`

In [23]:
print("[3,1,2,4]:", bug_default_zero([3, 1, 2, 4]), "| đúng:", Solution().sumSubarrayMins([3, 1, 2, 4]))
assert bug_default_zero([3, 1, 2, 4]) != 17

[3,1,2,4]: 11 | đúng: 17


**Vì sao sai:** nếu không có phần tử nhỏ hơn bên trái, đoạn có thể bắt đầu từ chỉ số `0`, tức `L = -1` (`i - L = i + 1` cách chọn). **Đúng:** `-1` (và `n` cho biên phải).

### Lỗi 4: quên modulo, hoặc tràn 32-bit ở ngôn ngữ khác

In [24]:
print("không mod:", bug_no_mod(big), "| đúng:", Solution().sumSubarrayMins(big))
print("int32    :", bug_int32(big))

không mod: 13500450000000 | đúng: 449905500
int32    : 367788665


**Vì sao sai:** đáp án thật tới `~10^13` (lớn hơn `MOD`); ở Java/C++ phép nhân `arr[j]·(j−left)·(i−j)` có thể tới `3·10^4·3·10^4·3·10^4 ≈ 2.7·10^13` — **tràn `int`**. **Đúng:** dùng `long` và `% MOD` sau mỗi cộng/nhân.

### Lỗi 5 (DP): quên cộng phần `dp[p]`

In [25]:
bad = next(a for a in tests if bug_dp_no_carry(a) != oracle(a))
print(bad, "->", bug_dp_no_carry(bad), "| đúng:", oracle(bad))

[2, 3] -> 5 | đúng: 7


**Vì sao sai:** các đoạn bắt đầu ở `≤ p` cũng kết thúc tại `i` và có min bằng min của đoạn tương ứng kết thúc tại `p`; tổng của chúng là `dp[p]`. **Đúng:** `dp[i] = dp[p] + arr[i]·(i − p)`. (Ghi chú: ở công thức DP, chọn `p` là gần nhất `<` hay `≤` đều cho kết quả đúng; sự bất đối xứng chỉ bắt buộc với cách **đếm số đoạn** `(i − L)(R − i)`.)

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tổng/số lượng liên quan tới **min hoặc max của mọi đoạn con**" → kỹ thuật **đóng góp** + **ngăn xếp đơn điệu** để tìm biên trái/phải của mỗi phần tử.

**Mẹo:**

1. Hoán đổi: cộng theo **phần tử** thay vì theo đoạn.
2. Xác định biên bằng ngăn xếp đơn điệu; **ghi nhớ quy ước** `<` một bên, `≤` bên kia.
3. Phần tử gác (`0` hoặc `-∞`) ở cuối để pop hết.
4. Biên mặc định: `-1` bên trái, `n` bên phải.
5. Làm việc với `long` và `% MOD`.
6. Oracle: `min` của mọi đoạn con.

**Họ bài cùng pattern:** Sum of Subarray Ranges (2104), Largest Rectangle in Histogram (84), Maximal Rectangle (85), Sum of Total Strength of Wizards (2281), Online Stock Span (901), Next Greater Element (496), Count Subarrays Where Max Element Appears at Least K Times.

## 15. Biến thể

1. **Tổng max của mọi đoạn con:** đối xứng (đổi dấu so sánh).
2. **Sum of Subarray Ranges (2104):** `Σ(max − min)` = tổng max − tổng min.
3. **Largest Rectangle in Histogram (84):** diện tích lớn nhất = `max_i arr[i] · (R[i] − L[i] − 1)` với biên **nghiêm ngặt cả hai** (chỉ cần biên).
4. **Đếm số đoạn con có min bằng `x`.**
5. **Tổng min của đoạn con độ dài đúng `k`.**

Cài biến thể 1–3 và đối chiếu vét cạn:

In [26]:
def sum_maxs(arr):
    return -sum_mins_exact([-x for x in arr])             # max = -min(-x); dùng bản chính xác không modulo


def sum_mins_exact(arr):
    return exact_sum_signed(arr)


def exact_sum_signed(a):
    """Như exact_sum nhưng cho phép giá trị âm (dùng DP theo điểm cuối)."""
    dp, st, total = [0] * len(a), [], 0
    for i, x in enumerate(a):
        while st and a[st[-1]] >= x:
            st.pop()
        p = st[-1] if st else -1
        dp[i] = (dp[p] if p >= 0 else 0) + x * (i - p)
        total += dp[i]
        st.append(i)
    return total


def sum_ranges(arr):
    return sum_maxs(arr) - exact_sum_signed(arr)


def sum_ranges_brute(arr):
    n = len(arr)
    return sum(max(arr[i:j + 1]) - min(arr[i:j + 1]) for i in range(n) for j in range(i, n))


def largest_rectangle(heights):
    L, R = boundaries(heights)                            # L: gần nhất bên trái < ; R: gần nhất bên phải <= (với biên >= vẫn cho diện tích đúng)
    best = 0
    for i, h in enumerate(heights):
        best = max(best, h * (R[i] - L[i] - 1))
    return best


def largest_rectangle_brute(heights):
    n = len(heights)
    return max(min(heights[i:j + 1]) * (j - i + 1) for i in range(n) for j in range(i, n))


rng = random.Random(15)
for _ in range(500):
    a = [rng.randint(1, 9) for _ in range(rng.randint(1, 10))]
    assert sum_maxs(a) == sum(max(a[i:j + 1]) for i in range(len(a)) for j in range(i, len(a)))
    assert sum_ranges(a) == sum_ranges_brute(a)
    assert largest_rectangle(a) == largest_rectangle_brute(a)
assert sum_ranges([1, 2, 3]) == 4 and sum_ranges([1, 3, 3]) == 4 and largest_rectangle([2, 1, 5, 6, 2, 3]) == 10
print("tổng max, Sum of Subarray Ranges (2104), Largest Rectangle in Histogram (84): khớp vét cạn trên 500 mảng")

tổng max, Sum of Subarray Ranges (2104), Largest Rectangle in Histogram (84): khớp vét cạn trên 500 mảng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đoạn con | min chạy | `O(n²)` | `O(1)` | oracle |
| 2. Hai ngăn xếp | `L` (`<`), `R` (`≤`), đếm | `O(n)` | `O(n)` | rõ ràng |
| 3. Một lượt | tính khi pop, phần tử gác | `O(n)` | `O(n)` | **đáp án chính** |
| 4. DP theo điểm cuối | `dp[i] = dp[p] + a[i]·(i−p)` | `O(n)` | `O(n)` | một biên |

Cả bốn khớp oracle mọi-đoạn-con trên 10 500 mảng nhỏ (gồm mọi hoán vị của `1..7`) và khớp nhau trên `n = 3·10^4` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [27]:
print(Solution().sumSubarrayMins([1, 2, 3]), Solution().sumSubarrayMins([2, 2, 2]), Solution().sumSubarrayMins([4, 1, 4]), sum_ranges([1, 3, 3]))

10 12 12 4


### Bài 2 (tính tay)

`arr = [2, 4, 1, 3]`. (a) Với mỗi `i`, tính `L[i]` (`<`), `R[i]` (`≤`) và `cnt[i]`. (b) Tổng `Σ arr[i]·cnt[i]`? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách một lượt:

In [28]:
def one_pass_todo(arr):
    total, st = 0, []
    for i in range(len(arr) + 1):
        cur = arr[i] if i < len(arr) else 0
        while st and arr[st[-1]] >= cur:
            j = st.pop()
            pass        # TODO 1: left = đỉnh còn lại (hoặc -1)
            pass        # TODO 2: total += arr[j] * (j - left) * (i - j)
        st.append(i)
    return total % MOD


# Khi làm xong, bỏ comment để kiểm tra:
# assert [one_pass_todo(a) for a, _ in examples] == [17, 444]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DP theo điểm cuối**:

In [29]:
class MySolution:
    def sumSubarrayMins(self, arr):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().sumSubarrayMins(a) == w for a, w in examples)

### Bài 5 (mini challenge): Sum of Subarray Ranges (2104)

Tính `Σ (max(b) − min(b))` trên mọi đoạn con `b` bằng **hai** lần kỹ thuật đóng góp (tổng max trừ tổng min), `O(n)`. Kiểm tra bằng vét cạn.

In [30]:
def sum_subarray_ranges(nums):
    pass            # TODO


# Khi làm xong:
# assert sum_subarray_ranges([1, 2, 3]) == 4 and sum_subarray_ranges([4, -2, -3, 4, 1]) == 59

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: dùng công thức `Σ arr[i]·(i − L)(R − i)`.
- Mức 2: `[1,2,3]`: đoạn: `1,2,3,1,2,1` → `10`; `[2,2,2]`: `2·6 = 12`.
- Mức 3: kết quả `10 12 12 4`: `[4,1,4]` → `4 + 1 + 4 + 1 + 1 + 1 = 12`; `sum_ranges([1,3,3]) = 4`.

**Bài 2**

- Mức 1: `L` (nhỏ hơn hẳn bên trái), `R` (nhỏ hơn hoặc bằng bên phải).
- Mức 2: `i=0 (2)`: `L=-1`, `R=2` → `1·2 = 2`; `i=1 (4)`: `L=0`, `R=2` → `1·1 = 1`; `i=2 (1)`: `L=-1`, `R=4` → `3·2 = 6`; `i=3 (3)`: `L=2`, `R=4` → `1·1 = 1`.
- Mức 3: `2·2 + 4·1 + 1·6 + 3·1 = 17`.

**Bài 3**

- Mức 1: `left = st[-1] if st else -1`.
- Mức 2: `total += arr[j] * (j - left) * (i - j)`.
- Mức 3: xem Cách 3.

**Bài 4**

- Mức 1: `p` = gần nhất bên trái `<`.
- Mức 2: `dp[i] = dp[p] + arr[i] * (i - p)`.
- Mức 3: đáp án `Σ dp[i]`.

**Bài 5**

- Mức 1: `Σ(max − min) = Σ max − Σ min`.
- Mức 2: tổng max = `−` (tổng min của mảng đối dấu).
- Mức 3: dùng `exact_sum_signed` (hoặc viết lại) và trừ.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [31]:
assert (Solution().sumSubarrayMins([1, 2, 3]), Solution().sumSubarrayMins([2, 2, 2]), Solution().sumSubarrayMins([4, 1, 4]), sum_ranges([1, 3, 3])) == (10, 12, 12, 4)
assert all(oracle(a) == v for a, v in [([1, 2, 3], 10), ([2, 2, 2], 12), ([4, 1, 4], 12)])
print("Bài 1 OK: 10 12 12 4")

Bài 1 OK: 10 12 12 4


### Bài 2

`cnt = [2, 1, 6, 1]` ⇒ tổng `2·2 + 4·1 + 1·6 + 3·1 = 17`.

In [32]:
a = [2, 4, 1, 3]
L, R = boundaries(a)
cnt = [(i - L[i]) * (R[i] - i) for i in range(4)]
assert L == [-1, 0, -1, 2] and R == [2, 2, 4, 4] and cnt == [2, 1, 6, 1]
assert sum(x * c for x, c in zip(a, cnt)) == 17 == oracle(a)
print("Bài 2 OK:", L, R, cnt)

Bài 2 OK: [-1, 0, -1, 2] [2, 2, 4, 4] [2, 1, 6, 1]


### Bài 3

In [33]:
def one_pass_answer(arr):
    total, st = 0, []
    for i in range(len(arr) + 1):
        cur = arr[i] if i < len(arr) else 0
        while st and arr[st[-1]] >= cur:
            j = st.pop()
            left = st[-1] if st else -1
            total += arr[j] * (j - left) * (i - j)
        st.append(i)
    return total % MOD


assert [one_pass_answer(a) for a, _ in examples] == [17, 444]
for a in tests[:300]:
    assert one_pass_answer(a) == oracle(a)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [34]:
class MySolutionAnswer:
    def sumSubarrayMins(self, arr):
        n = len(arr)
        dp, st, total = [0] * n, [], 0
        for i in range(n):
            while st and arr[st[-1]] >= arr[i]:
                st.pop()
            p = st[-1] if st else -1
            dp[i] = (dp[p] if p >= 0 else 0) + arr[i] * (i - p)
            total += dp[i]
            st.append(i)
        return total % (10 ** 9 + 7)


assert all(MySolutionAnswer().sumSubarrayMins(a) == w for a, w in examples)
for a in tests[:300]:
    assert MySolutionAnswer().sumSubarrayMins(a) == oracle(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [35]:
def sum_subarray_ranges_answer(nums):
    return exact_sum_signed([-x for x in nums]) * -1 - exact_sum_signed(nums)


assert sum_subarray_ranges_answer([1, 2, 3]) == 4 and sum_subarray_ranges_answer([4, -2, -3, 4, 1]) == 59
rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(-5, 5) for _ in range(rng.randint(1, 10))]
    assert sum_subarray_ranges_answer(a) == sum_ranges_brute(a)
print("Bài 5 OK: khớp vét cạn (kể cả số âm)")

Bài 5 OK: khớp vét cạn (kể cả số âm)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tổng min/max của mọi đoạn con → đóng góp + ngăn xếp đơn điệu):**

```python
class Solution:
    def sumSubarrayMins(self, arr: List[int]) -> int:
        MOD = 10 ** 9 + 7
        total, st = 0, []
        for i in range(len(arr) + 1):
            cur = arr[i] if i < len(arr) else 0          # phần tử gác
            while st and arr[st[-1]] >= cur:
                j = st.pop()
                left = st[-1] if st else -1
                total += arr[j] * (j - left) * (i - j)
            st.append(i)
        return total % MOD
```

**Checklist trước khi submit:**

- [ ] Biên bất đối xứng: trái `<` (gần nhất nhỏ hơn hẳn), phải `≤` (gần nhất nhỏ hơn hoặc bằng) — không cả hai cùng `<` hay cùng `≤`?
- [ ] Biên mặc định `-1` (trái) và `n` (phải); phần tử gác ở cuối?
- [ ] Số đoạn `(i − L)(R − i)`; đóng góp `arr[i]·cnt`?
- [ ] `% (10**9 + 7)` ở cuối (và dùng `long` ở ngôn ngữ 32-bit)?
- [ ] Đã thử `[3,1,2,4] → 17`, `[11,81,94,43,3] → 444`, `[2,2] → 6`, mảng toàn bằng nhau?
- [ ] Đúng tên `class Solution`, `sumSubarrayMins(self, arr)`, không in debug.